# 05 · Qdrant collections `cse_units_v1` and `cse_win500_v1`

Plan section 3.5. Both collections have the same schema:

| named vector | type | config |
|---|---|---|
| `dense` | dense, 1024 | Cosine, HNSW default |
| `colbert` | multivector, 1024 per token | MAX_SIM, float16, `hnsw m=0` (no index) |
| `bge_sparse` | sparse | bge-m3 lexical weights, no modifier |
| `bm25` | sparse | `modifier=IDF` |

Payload: `unit_id, index, section_id, group_id, kind, subpart, engine_type, page_start, page_end, part_index, part_count, char_start, char_end, n_tokens_bge, breadcrumb, cross_refs, corpus_version, text` (+ `colbert_in_qdrant`, `n_colbert`). Point ids are `uuid5(unit_id)`, so re-ingestion is idempotent.

**Before running:** notebook 04 has written the embeddings; `docker compose up -d qdrant` (phase A: vLLM stays stopped). The encoder is not needed here.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [ ]:
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from csrag.corpus import COLLECTIONS, UNIT_FILES, Corpus
from csrag import index as ix
from csrag.embed import BM25, EmbeddingStore
from csrag.enrich import corpus_version
from qdrant_client import models as qm

corpus = Corpus(DATA)
assert corpus.has_windows, "run notebook 04 first"
client = ix.connect()
print("Qdrant", client.info().version)
stores = {i: EmbeddingStore(COLLECTIONS[i], DATA / "embeddings") for i in corpus.indexes()}
{i: s.meta.get("bm25") for i, s in stores.items()}

## Create and upsert

`RECREATE = True` drops and rebuilds a collection; otherwise upserts are idempotent.

In [ ]:
RECREATE = False
for index in corpus.indexes():
    units = list(corpus.units(index).values())
    assert not stores[index].missing(units), f"{index}: embeddings missing, run notebook 04"
    ix.create_collection(client, index, recreate=RECREATE)
    n = ix.upsert(client, index, units, stores[index], corpus_version(DATA / UNIT_FILES[index]), progress=tqdm)
    print(index, COLLECTIONS[index], n, "points upserted")

## Ingestion checks (plan 3.5)

* the point count equals the unit file's line count;
* every unit id is in Qdrant and in the JSONL corpus;
* a random vector round-trips exactly (dense within 1e-6, because Qdrant re-normalises cosine vectors);
* self-retrieval: each unit's own text retrieves itself at rank 1 with dense and BM25 (≥ 99%; for windows an overlapping neighbour at rank 1 also counts).

In [ ]:
checks = {}
for index in corpus.indexes():
    bm25 = BM25(avg_len=stores[index].meta["bm25"]["avg_len"])
    checks[index] = ix.verify(client, index, list(corpus.units(index).values()), stores[index], bm25, n_roundtrip=20, progress=tqdm)
pd.DataFrame({i: {k: v for k, v in c.items() if k != "self_fail"} for i, c in checks.items()})

In [ ]:
for i, c in checks.items():
    print(i, "self-retrieval failures:", {k: v[:5] for k, v in c["self_fail"].items()})
assert all(c["ok"] for c in checks.values()), "ingestion checks failed"

## HNSW vs exact dense search

With 252 and ~440 points the ANN error should be nil. Here every unit's own dense vector is a query; notebook 08 repeats the check with the questions.

In [ ]:
rng = random.Random(0)
rows = []
for index in corpus.indexes():
    ids = sorted(corpus.units(index))
    same = 0
    for uid in rng.sample(ids, 50):
        q = stores[index].get(uid)["dense"].tolist()
        a = [p.id for p in client.query_points(COLLECTIONS[index], query=q, using="dense", limit=50).points]
        b = [p.id for p in client.query_points(COLLECTIONS[index], query=q, using="dense", limit=50,
                                               search_params=qm.SearchParams(exact=True)).points]
        same += a == b
    rows.append({"index": index, "queries": 50, "identical_top50": same})
pd.DataFrame(rows)

## ColBERT engines agree

Qdrant caps a multivector at 1023 vectors, so long base units are stored without their ColBERT vector and C1/M1 compute MaxSim client-side (`ColbertIndex`). On the points that do hold a ColBERT vector in Qdrant, both engines must give the same scores. Queries: the first 32 token vectors of random units.

In [ ]:
from csrag.retrievers.qdrant_search import ColbertIndex
rows = []
for index in corpus.indexes():
    ci = ColbertIndex(stores[index], corpus.units(index))
    assert not ci.missing
    qs = [stores[index].get(u)["colbert"][:32] for u in rng.sample(sorted(corpus.units(index)), 8)]
    rows.append({"index": index, **ix.check_colbert_engines(client, index, ci, qs)})
pd.DataFrame(rows)

## Alternative evaluated: Qdrant's server-side BM25

Qdrant ≥ 1.15 can compute BM25 itself (`models.Document(text, model="qdrant/bm25")`), which would make the BM25 parameters part of the collection. On v1.18.3 it produces the same token ids and weights as fastembed. This builds a throw-away collection with server-side BM25 for the base units and compares the top-10 of 30 unit texts with the fastembed-based `bm25` vector. fastembed stays the method of record.

In [ ]:
TMP = "tmp_bm25_server_check"
units = list(corpus.units("units").values())
avg_len = stores["units"].meta["bm25"]["avg_len"]
opts = {"avg_len": avg_len, "k": 1.2, "b": 0.75}
if client.collection_exists(TMP):
    client.delete_collection(TMP)
client.create_collection(TMP, vectors_config={}, sparse_vectors_config={"bm25": qm.SparseVectorParams(modifier=qm.Modifier.IDF)})
client.upsert(TMP, points=[qm.PointStruct(id=ix.point_id(u["unit_id"]), payload={"unit_id": u["unit_id"]},
                                          vector={"bm25": qm.Document(text=u["text"], model="qdrant/bm25", options=opts)}) for u in units])
bm25 = BM25(avg_len=avg_len)
same = 0
for u in rng.sample(units, 30):
    s = bm25.embed_query(u["text"])
    a = [p.payload["unit_id"] for p in client.query_points(COLLECTIONS["units"], query=qm.SparseVector(indices=s.indices.tolist(), values=s.values.tolist()), using="bm25", limit=10, with_payload=["unit_id"]).points]
    b = [p.payload["unit_id"] for p in client.query_points(TMP, query=qm.Document(text=u["text"], model="qdrant/bm25"), using="bm25", limit=10, with_payload=["unit_id"]).points]
    same += a == b
client.delete_collection(TMP)
print(f"identical top-10 for {same}/30 queries")

## In-memory corpus matches the files

In [ ]:
counts = {
    "sections": (len(corpus.sections), 175),
    "groups": (len(corpus.groups), 92),
    "base units": (len(corpus.units("units")), 252),
    "windows": (len(corpus.units("win500")), sum(1 for _ in open(DATA / "windows_500.jsonl"))),
    "points cse_units_v1": (client.get_collection(COLLECTIONS["units"]).points_count, 252),
    "points cse_win500_v1": (client.get_collection(COLLECTIONS["win500"]).points_count, len(corpus.units("win500"))),
}
df = pd.DataFrame(counts, index=["found", "expected"]).T
assert (df["found"] == df["expected"]).all()
df